# Market Basket PiepeLine For Analaysis

## Author : Es-saouiqui Amine

> - #### In this notebook we will create a simple data pipeline,starting from transactions extraction in RDBMS (MySQL in ourcase) through  uncovering the relationship between products to  load the results in a great graphical representative structure 

### 1-Setting up and configuring the work environement  

In [1]:
# Import necessary modules
from pyspark.sql import SparkSession,DataFrame

In [2]:
# Define connector driver paths for jdbc usage 
from pathlib import Path
mysql_connector_path:Path=Path("/usr/share/java/mysql-connector-java-8.0.23.jar")
neo4j_connector_path="org.neo4j:neo4j-connector-apache-spark_2.12:5.3.2_for_spark_3"

In [3]:
# set a SparkSession
# Attention := Due to test and exploring purposes, we just used default and minimal computionnal ressources,we recommand to add 
# more configurations in case!
spark :SparkSession=SparkSession.builder\
.appName("BasketMarket")\
.config("spark.jars",mysql_connector_path)\
.config("spark.jars.packages",neo4j_connector_path)\
.getOrCreate()


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/azureuser/.ivy2/cache
The jars for the packages stored in: /home/azureuser/.ivy2/jars
org.neo4j#neo4j-connector-apache-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f2e370d1-1ee7-47ce-ac8f-328168e6323a;1.0
	confs: [default]
	found org.neo4j#neo4j-connector-apache-spark_2.12;5.3.2_for_spark_3 in central
	found org.neo4j#neo4j-connector-apache-spark_2.12_common;5.3.2_for_spark_3 in central
	found org.neo4j#neo4j-cypher-dsl;2022.11.0 in central
	found org.apiguardian#apiguardian-api;1.1.2 in central
	found org.neo4j.driver#neo4j-java-driver;4.4.18 in central
	found org.reactivestreams#reactive-streams;1.0.4 in central
:: resolution report :: resolve 430ms :: artifacts dl 14ms
	:: modules in use:
	org.apiguardian#apiguardian-api;1.1.2 from central in [default]
	org.neo4j#neo4j-connector-apache-spark_2.12;5.3.2_for_spark_3 from central in [default]
	org.neo4j#neo4j-connector-apache-spark_2.12_common;5.3.2_f

### 2- Extraction

In [4]:
# Set MySQL url 
mysql_url:str="jdbc:mysql://trans1.mysql.database.azure.com:3306/transaction_db"
properties:dict={
    "user":"azure1",
    "password":"mine-2003",
    "driver":"com.mysql.cj.jdbc.Driver"
}

In [5]:
# Load data table into DataFrame
df:DataFrame=spark\
.read\
.jdbc(url=mysql_url,table="transactions",properties=properties)

In [6]:
# Show The data Frame 
df.show(5)

+----+--------------------+
|  id|            products|
+----+--------------------+
|9836|rolls/buns,bottle...|
|9837|whole milk,brown ...|
|9838|root vegetables,b...|
|9839|white bread,pastr...|
|9840|nuts/prunes,whole...|
+----+--------------------+
only showing top 5 rows



In [7]:
# Describe the Schema 
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- products: string (nullable = true)



In [8]:
# Drop null values
df:DataFrame=df.dropna()

### 3. Transformation

>- As we can see each row represent a transaction and each column represents the purchased products in this transaction,
>   each purchased products are formated in a single string distinguished by ','.\
>  As dedecuction,each line should be transformed into a list of products

In [9]:
# To do so
from pyspark.sql.functions import split 
"""
split -> Column :
return a column out of a string column where each row 
contains a list of word that get splitted based on a regular expression patttern
"""
df=df.withColumn('list_of_products',split(df['products'],','))

In [10]:
df.show(5)

+----+--------------------+--------------------+
|  id|            products|    list_of_products|
+----+--------------------+--------------------+
|9836|rolls/buns,bottle...|[rolls/buns, bott...|
|9837|whole milk,brown ...|[whole milk, brow...|
|9838|root vegetables,b...|[root vegetables,...|
|9839|white bread,pastr...|[white bread, pas...|
|9840|nuts/prunes,whole...|[nuts/prunes, who...|
+----+--------------------+--------------------+
only showing top 5 rows



## 4. Basket Market Analysis 
The main purpose from this analysis,is to identify relationships between products frequently purshased together.It analyzes transaction data to 
uncover patterns,sush as which products are commonly bought as agroup which gives an insightful vision for make decision to enhance buiseness 

>- Question : How to detect the commonly bought products??

#### To answer this question you have to know about the:
-Support : is a measure of how frequently an itemset appears in the dataset.\
-Confidence: is measure defined as the probability that an item A is purshased when an item B is purshased,which means it measures  the strength associations 

Happily,there is many algorithm that can solve our problem based on these two measure.
The one that will we use here is Fp-Growth Algorithm,this model returs the most frequent itemsets in the dataset.\
It returns also the association rules : which the probablistic relationships between items in transactions,with metrics like support,confidencd,etc...

In [11]:
from pyspark.ml.fpm import FPGrowth
items=df.select('list_of_products')

In [12]:
items.show(5,truncate=False)

+-------------------------------------------------------------------------------------------------------------------+
|list_of_products                                                                                                   |
+-------------------------------------------------------------------------------------------------------------------+
|[rolls/buns, bottled water, liquor]                                                                                |
|[whole milk, brown bread]                                                                                          |
|[root vegetables, butter milk, specialty cheese, specialty bar]                                                    |
|[white bread, pastry, roll products, soda, fruit/vegetable juice, long life bakery product, waffles, shopping bags]|
|[nuts/prunes, whole milk, hard cheese, processed cheese, newspapers]                                               |
+-------------------------------------------------------

In [13]:
# Define the model
Fpg=FPGrowth(itemsCol='list_of_products',minSupport=0.01,minConfidence=0.1)

In [14]:
Fpg

FPGrowth_ccc2aee8f55f

In [15]:
# Apply the algorithm 
result=Fpg.fit(items)
result

FPGrowthModel: uid=FPGrowth_ccc2aee8f55f, numTrainingRecords=393

In [16]:
# Show the frequent itemsets
freqitems=result.freqItemsets
freqitems.show(5)

+--------------------+----+
|               items|freq|
+--------------------+----+
|[chocolate marshm...|   4|
| [canned vegetables]|   5|
|       [frankfurter]|  22|
|[frankfurter, sho...|   4|
|[frankfurter, rol...|   9|
+--------------------+----+
only showing top 5 rows



In [17]:
# Show association rules 
from pyspark.sql.functions import size,col
association_rules=result.associationRules
association_rules.show(5)

+--------------------+------------+------------------+------------------+--------------------+
|          antecedent|  consequent|        confidence|              lift|             support|
+--------------------+------------+------------------+------------------+--------------------+
|[waffles, rolls/b...|[whole milk]|             0.625|2.2743055555555554| 0.01272264631043257|
|[shopping bags, p...|[whole milk]|0.7142857142857143|2.5992063492063493| 0.01272264631043257|
|   [dessert, pastry]|[whole milk]|0.6666666666666666|2.4259259259259256|0.010178117048346057|
|[beef, whipped/so...|[whole milk]|               1.0| 3.638888888888889|0.010178117048346057|
|          [cake bar]|   [dessert]|0.6666666666666666|16.374999999999996|0.010178117048346057|
+--------------------+------------+------------------+------------------+--------------------+
only showing top 5 rows



## 5 . Loading 
Well after detecting the frequent Items And association rules.We need to store these results in a database.Well modeling a structering schema to store
data will make Analysing  very complex especially because  RDBMS forbid data composition.wich makes relation very complex between entities.
As a good approach,thanks to graph theory we can represent FrequentItem as node And association rules as relationship

### Node Modeling :
> #### Product :
        >id.\
        >name
> #### FreqItems:
        >id\
        >frequence
### Relationship Modeling:
> #### ASSOCIATED \[FreqItems <-> FreqItems]
        > support.\
        > confidence.\
        > lift .
> #### CONTAINS \[FreqItems -> Product]
        > id

In [18]:
# Get all products 
from pyspark.sql.functions import explode
products=items.select(explode(items.list_of_products).alias('product')).distinct()
products.show(5)

[Stage 13:>                                                         (0 + 1) / 1]

+------------------+
|           product|
+------------------+
|    snack products|
|         beverages|
|pickled vegetables|
|           vinegar|
|            dishes|
+------------------+
only showing top 5 rows



In [19]:
# Generate the frequ item id 
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
generate_itemsets_id=udf(lambda itemset:'-'.join(sorted(itemset)),StringType())

In [20]:
itemets=freqitems.withColumn("id",generate_itemsets_id(freqitems.items))

In [21]:
itemets.show(5)

[Stage 17:>                                                         (0 + 1) / 1]

+--------------------+----+--------------------+
|               items|freq|                  id|
+--------------------+----+--------------------+
|[chocolate marshm...|   4|chocolate marshma...|
| [canned vegetables]|   5|   canned vegetables|
|       [frankfurter]|  22|         frankfurter|
|[frankfurter, sho...|   4|frankfurter-shopp...|
|[frankfurter, rol...|   9|frankfurter-rolls...|
+--------------------+----+--------------------+
only showing top 5 rows



In [22]:
itemsets_with_products=itemets.withColumn("product",explode(itemets.items)).select("id","product","freq")
itemsets_with_products.show(5,truncate=False)

+-------------------------+---------------------+----+
|id                       |product              |freq|
+-------------------------+---------------------+----+
|chocolate marshmallow    |chocolate marshmallow|4   |
|canned vegetables        |canned vegetables    |5   |
|frankfurter              |frankfurter          |22  |
|frankfurter-shopping bags|frankfurter          |4   |
|frankfurter-shopping bags|shopping bags        |4   |
+-------------------------+---------------------+----+
only showing top 5 rows



In [23]:
association_rule_ids=association_rules\
.withColumn('ant_id',generate_itemsets_id(association_rules.antecedent))\
.withColumn('con_id',udf(lambda x:x[0],StringType())(col('consequent')))

In [24]:
association_rule_ids.show(5)

+--------------------+------------+------------------+------------------+--------------------+--------------------+----------+
|          antecedent|  consequent|        confidence|              lift|             support|              ant_id|    con_id|
+--------------------+------------+------------------+------------------+--------------------+--------------------+----------+
|[waffles, rolls/b...|[whole milk]|             0.625|2.2743055555555554| 0.01272264631043257|  rolls/buns-waffles|whole milk|
|[shopping bags, p...|[whole milk]|0.7142857142857143|2.5992063492063493| 0.01272264631043257|pastry-shopping bags|whole milk|
|   [dessert, pastry]|[whole milk]|0.6666666666666666|2.4259259259259256|0.010178117048346057|      dessert-pastry|whole milk|
|[beef, whipped/so...|[whole milk]|               1.0| 3.638888888888889|0.010178117048346057|beef-whipped/sour...|whole milk|
|          [cake bar]|   [dessert]|0.6666666666666666|16.374999999999996|0.010178117048346057|            cake 

In [25]:
association_rules_to_load=association_rule_ids.select('ant_id','con_id','confidence','lift','support')
association_rules_to_load.show(5)

+--------------------+----------+------------------+------------------+--------------------+
|              ant_id|    con_id|        confidence|              lift|             support|
+--------------------+----------+------------------+------------------+--------------------+
|  rolls/buns-waffles|whole milk|             0.625|2.2743055555555554| 0.01272264631043257|
|pastry-shopping bags|whole milk|0.7142857142857143|2.5992063492063493| 0.01272264631043257|
|      dessert-pastry|whole milk|0.6666666666666666|2.4259259259259256|0.010178117048346057|
|beef-whipped/sour...|whole milk|               1.0| 3.638888888888889|0.010178117048346057|
|            cake bar|   dessert|0.6666666666666666|16.374999999999996|0.010178117048346057|
+--------------------+----------+------------------+------------------+--------------------+
only showing top 5 rows



In [26]:
#Neo4j config
neo4j_config={
    "url":"bolt://68.221.26.50:7687",
    "authentication.basic.username":"neo4j",
    "authentication.basic.password":"amine2003",
    "authentication.type":"basic",
    "encrypted":"false"
}

In [27]:
products=products.select(products.product.alias("product_name"))

In [55]:
products.write\
.format('org.neo4j.spark.DataSource')\
.options(**neo4j_config)\
.option("labels","Product")\
.option("node.keys","product_name")\
.mode("overwrite")\
.save()

In [28]:
ItemSets=itemsets_with_products.select(col("id").alias("itemset_name"),col("freq").alias("frequence")).distinct()
ItemSets.show()

+--------------------+---------+
|        itemset_name|frequence|
+--------------------+---------+
|     butter milk-ham|        4|
|newspapers-tropic...|        6|
|bottled water-oth...|        8|
|bottled beer-whol...|        7|
|              yogurt|       52|
|dishes-other vege...|        4|
|bottled water-new...|        7|
|butter-domestic e...|        4|
|      flower (seeds)|        6|
|other vegetables-...|       11|
|bottled water-whi...|        8|
|             napkins|       24|
|brown bread-butte...|        4|
|napkins-root vege...|        4|
|   bottled beer-soda|        6|
|napkins-rolls/bun...|        4|
|         white bread|       17|
|  domestic eggs-soda|        5|
|      ham-whole milk|        8|
|    hygiene articles|       17|
+--------------------+---------+
only showing top 20 rows



In [56]:
ItemSets.write\
.format('org.neo4j.spark.DataSource')\
.options(**neo4j_config)\
.option("labels","FrequentItems")\
.option("node.keys","itemset_name")\
.mode("overwrite")\
.save()

In [29]:
from pyspark.sql.functions import monotonically_increasing_id
contains_df=itemsets_with_products.select(col("id").alias("itemset_name"),col("product").alias("product_name"))
# .withColumn('rel.id',monotonically_increasing_id())
contains_df.show(5)
contains_df.printSchema()

+--------------------+--------------------+
|        itemset_name|        product_name|
+--------------------+--------------------+
|chocolate marshma...|chocolate marshma...|
|   canned vegetables|   canned vegetables|
|         frankfurter|         frankfurter|
|frankfurter-shopp...|         frankfurter|
|frankfurter-shopp...|       shopping bags|
+--------------------+--------------------+
only showing top 5 rows

root
 |-- itemset_name: string (nullable = true)
 |-- product_name: string (nullable = false)



In [30]:
from pyspark.sql.functions import lit
contains_df_with_labels = contains_df.withColumnRenamed("itemset_name", "source.id") \
                                     .withColumnRenamed("product_name", "target.id") \
                                     .withColumn("relationship.type", lit("CONTAINS"))
contains_df_with_labels.show(5)

+--------------------+--------------------+-----------------+
|           source.id|           target.id|relationship.type|
+--------------------+--------------------+-----------------+
|chocolate marshma...|chocolate marshma...|         CONTAINS|
|   canned vegetables|   canned vegetables|         CONTAINS|
|         frankfurter|         frankfurter|         CONTAINS|
|frankfurter-shopp...|         frankfurter|         CONTAINS|
|frankfurter-shopp...|       shopping bags|         CONTAINS|
+--------------------+--------------------+-----------------+
only showing top 5 rows



In [37]:
contains_df_with_labels.write\
.format('org.neo4j.spark.DataSource')\
.options(**neo4j_config)\
.option("relationship","CONTAINS")\
.option("relationship.source.labels",":FrequentItems")\
.option("relationship.target.labels",":Product")\
.mode("overwrite")\
.save()

In [31]:
ps=spark.read.format('org.neo4j.spark.DataSource').options(**neo4j_config).option("labels","FrequentItems").load()
                                                                                

24/12/10 21:32:31 WARN SchemaService: Switching to query schema resolution


In [32]:
ps.show()

+----+---------------+--------------------+---------+
|<id>|       <labels>|        itemset_name|frequence|
+----+---------------+--------------------+---------+
|   0|[FrequentItems]|specialty bar-who...|        5|
|   1|[FrequentItems]|   butter-rolls/buns|       10|
|   2|[FrequentItems]|butter-root veget...|        5|
|   3|[FrequentItems]|newspapers-rolls/...|        4|
|   4|[FrequentItems]|other vegetables-...|        4|
|   5|[FrequentItems]|             dessert|       16|
|   6|[FrequentItems]|dessert-pastry-wh...|        4|
|   7|[FrequentItems]|fruit/vegetable j...|        5|
|   8|[FrequentItems]|    specialty cheese|        4|
|   9|[FrequentItems]|domestic eggs-roo...|        5|
|  10|[FrequentItems]|bottled water-fru...|        6|
|  11|[FrequentItems]|  napkins-whole milk|       13|
|  12|[FrequentItems]|      pip fruit-soda|        7|
|  13|[FrequentItems]|rolls/buns-waffle...|        5|
|  14|[FrequentItems]|fruit/vegetable j...|        4|
|  15|[FrequentItems]|pip fr

In [33]:
psd=spark.read.format('org.neo4j.spark.DataSource').options(**neo4j_config).option("labels","Product").load()
psd.show(5)

24/12/10 21:32:40 WARN SchemaService: Switching to query schema resolution


+----+---------+------------------+
|<id>| <labels>|      product_name|
+----+---------+------------------+
| 136|[Product]|    snack products|
| 137|[Product]|         beverages|
| 138|[Product]|pickled vegetables|
| 139|[Product]|           vinegar|
| 140|[Product]|            dishes|
+----+---------+------------------+
only showing top 5 rows



In [34]:
new_df=ps.withColumn("products",split(col("itemset_name").alias("product"),'-'))
new_df.show(5,truncate=False)

+----+---------------+-----------------------------------------+---------+---------------------------------------------+
|<id>|<labels>       |itemset_name                             |frequence|products                                     |
+----+---------------+-----------------------------------------+---------+---------------------------------------------+
|0   |[FrequentItems]|specialty bar-whole milk                 |5        |[specialty bar, whole milk]                  |
|1   |[FrequentItems]|butter-rolls/buns                        |10       |[butter, rolls/buns]                         |
|2   |[FrequentItems]|butter-root vegetables                   |5        |[butter, root vegetables]                    |
|3   |[FrequentItems]|newspapers-rolls/buns-whole milk         |4        |[newspapers, rolls/buns, whole milk]         |
|4   |[FrequentItems]|other vegetables-pip fruit-tropical fruit|4        |[other vegetables, pip fruit, tropical fruit]|
+----+---------------+----------

In [36]:
dfs=new_df.select(col('<id>'),col('itemset_name'),explode(col('products')).alias("product"),col('<labels>'))
dfs.show(5)

+----+--------------------+-------------+---------------+
|<id>|        itemset_name|      product|       <labels>|
+----+--------------------+-------------+---------------+
|   0|specialty bar-who...|specialty bar|[FrequentItems]|
|   0|specialty bar-who...|   whole milk|[FrequentItems]|
|   1|   butter-rolls/buns|       butter|[FrequentItems]|
|   1|   butter-rolls/buns|   rolls/buns|[FrequentItems]|
|   2|butter-root veget...|       butter|[FrequentItems]|
+----+--------------------+-------------+---------------+
only showing top 5 rows



In [49]:
pd1=psd.withColumnRenamed('<id>','target.id')
pd=dfs.join(pd1,(dfs.product==pd1.product_name)).withColumnRenamed('<id>','source.id')

In [50]:
pd.show()

+---------+--------------------+--------------+---------------+---------+---------+--------------+
|source.id|        itemset_name|       product|       <labels>|target.id| <labels>|  product_name|
+---------+--------------------+--------------+---------------+---------+---------+--------------+
|      464|           beverages|     beverages|[FrequentItems]|      137|[Product]|     beverages|
|      509|beverages-other v...|     beverages|[FrequentItems]|      137|[Product]|     beverages|
|      541|             vinegar|       vinegar|[FrequentItems]|      139|[Product]|       vinegar|
|       53|              dishes|        dishes|[FrequentItems]|      140|[Product]|        dishes|
|       69|      red/blush wine|red/blush wine|[FrequentItems]|      142|[Product]|red/blush wine|
|      277|dishes-other vege...|        dishes|[FrequentItems]|      140|[Product]|        dishes|
|      308|red/blush wine-wh...|red/blush wine|[FrequentItems]|      142|[Product]|red/blush wine|
|      446

In [51]:
pd=pd.dropDuplicates(['itemset_name','product_name'])

In [52]:
pd.show()

+---------+--------------------+--------------------+---------------+---------+---------+--------------------+
|source.id|        itemset_name|             product|       <labels>|target.id| <labels>|        product_name|
+---------+--------------------+--------------------+---------------+---------+---------+--------------------+
|      405|Instant food prod...|Instant food prod...|[FrequentItems]|      198|[Product]|Instant food prod...|
|       45|UHT-milk-bottled ...|       bottled water|[FrequentItems]|      190|[Product]|       bottled water|
|      437|     UHT-milk-coffee|              coffee|[FrequentItems]|      246|[Product]|              coffee|
|      624|                beef|                beef|[FrequentItems]|      210|[Product]|                beef|
|       51|beef-other vegeta...|                beef|[FrequentItems]|      210|[Product]|                beef|
|       51|beef-other vegeta...|    other vegetables|[FrequentItems]|      189|[Product]|    other vegetables|
|

In [59]:
ps.write.format('org.neo4j.spark.DataSource')\
.options(**neo4j_config)\
.option("query","""MATCH (f:FrequentItems), (p:Product)
WHERE any(product IN split(f.itemset_name, '-') WHERE product = p.product_name)
MERGE (f)-[:CONTAINS]->(p)""")\
.mode("overwrite")\
.save()
# .option("relationship.source.node.keys",'itemset_name')\ # .option("relationship.target.node.keys","product_name")\

24/12/10 23:47:42 WARN Neo4jPartitionReader: Cannot close InternalTransaction because of the following exception:
org.neo4j.driver.exceptions.ConnectionReadTimeoutException: Connection read timed out due to it taking longer than the server-supplied timeout value via configuration hint.
	at org.neo4j.driver.internal.util.Futures.blockingGet(Futures.java:111)
	at org.neo4j.driver.internal.InternalTransaction.close(InternalTransaction.java:51)
	at org.neo4j.spark.util.Neo4jUtil$.closeSafely(Neo4jUtil.scala:65)
	at org.neo4j.spark.reader.BasePartitionReader.close(BasePartitionReader.scala:116)
	at org.apache.spark.sql.execution.datasources.v2.DataSourceRDD$$anon$1.$anonfun$advanceToNextIter$1(DataSourceRDD.scala:94)
	at org.apache.spark.sql.execution.datasources.v2.DataSourceRDD$$anon$1.$anonfun$advanceToNextIter$1$adapted(DataSourceRDD.scala:89)
	at org.apache.spark.TaskContext$$anon$1.onTaskCompletion(TaskContext.scala:137)
	at org.apache.spark.TaskContextImpl.$anonfun$invokeTaskCompleti